In [2]:
import os
import glob
import pyspark

path = 'c:\\Windows'
extension = '.csv'
e_len = len(extension)
os.chdir(path)
dfs = {}
result = glob.glob(f'*{extension}')
spark = pyspark.sql.SparkSession.builder.appName("pysaprk_python").getOrCreate()
for file in result:
    file_name = file[:-e_len].lower()
    dfs[file_name] = spark.read.csv(file, header=True, inferSchema=True)
    dfs[file_name].createOrReplaceTempView(file_name)


In [3]:
spark.catalog.listTables()

[Table(name='assessments', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='banking', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='courses', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='studentassessment', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='studentinfo', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='studentregistration', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='studentvle', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='vle', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [4]:
# clicks = spark.sql('select id_student, round(avg(sum_click), 2) as average_clicks from studentvle group by id_student')
# spark.sql('select gender, avg(average_clicks) average_clicks from studentinfo inner join clicks using(id_student) group by gender').show()
clicks = dfs["studentvle"].select('id_student', 'sum_click').groupby('id_student').agg({'sum_click': 'avg'}) \
    .withColumnRenamed("avg(sum_click)", "average_clicks")
clicks.createOrReplaceTempView('clicks')
dfs["studentinfo"].select('id_student', 'gender').join(clicks, on='id_student', how='inner').groupby('gender').agg({'average_clicks': 'avg'}).withColumnRenamed('avg(average_clicks)', 'average_clicks').show()




+------+-----------------+
|gender|   average_clicks|
+------+-----------------+
|     F|3.075360457816899|
|     M|3.470720052027121|
+------+-----------------+



In [5]:
# spark.sql("select highest_education, round(avg(average_clicks), 2) average_clicks from studentinfo inner join clicks using(id_student) group by highest_education").show()

dfs["studentinfo"].select('id_student', 'highest_education').join(clicks, on='id_student', how='inner').groupby('highest_education').agg({'average_clicks': 'avg'}).withColumnRenamed('avg(average_clicks)', 'average_clicks').show()

+--------------------+------------------+
|   highest_education|    average_clicks|
+--------------------+------------------+
|A Level or Equiva...| 3.294278176121476|
|  Lower Than A Level| 3.248515845235784|
|     No Formal quals|3.1358571080613524|
|Post Graduate Qua...|3.5277264021453063|
|    HE Qualification| 3.425328580875667|
+--------------------+------------------+



In [6]:
spark.sql('select imd_band, round(avg(average_clicks), 2) average_clicks from studentinfo inner join clicks using(id_student) group by imd_band order by average_clicks desc').show()

+--------+--------------+
|imd_band|average_clicks|
+--------+--------------+
|    NULL|          3.46|
| 90-100%|          3.39|
|  80-90%|          3.38|
|  70-80%|          3.34|
|  50-60%|          3.33|
|  30-40%|          3.32|
|  60-70%|           3.3|
|  40-50%|          3.27|
|   10-20|          3.26|
|  20-30%|          3.22|
|   0-10%|          3.15|
+--------+--------------+



In [7]:
# spark.sql('SELECT a.code_module, b.code_module, a.code_presentation, b.code_presentation, count(*) AS num_students \
# FROM studentregistration \
# a JOIN studentregistration b using(id_student) \
# WHERE a.date_unregistration IS NULL AND a.code_module < b.code_module AND a.code_presentation != b.code_presentation \
# GROUP BY a.code_module, b.code_module, a.code_presentation, b.code _presentation \
# ORDER BY num_students DESC').show()
from pyspark.sql.functions import col

dfs['studentregistration'].select('id_student', 'code_module', 'code_presentation').alias("first").join(dfs['studentregistration'].alias("second"),'id_student', 'inner').where((col('date_unregistration').isNull() & (col('first.code_module') < col('second.code_module')) & (col('first.code_presentation') != col('second.code_presentation')))). \
    groupby(['first.code_module', 'second.code_module', 'first.code_presentation', 'second.code_presentation']).count().withColumnRenamed('count', 'num_students').sort(col('num_students').desc()).show()

+-----------+-----------+-----------------+-----------------+------------+
|code_module|code_module|code_presentation|code_presentation|num_students|
+-----------+-----------+-----------------+-----------------+------------+
|        CCC|        EEE|            2014B|            2013J|         216|
|        CCC|        DDD|            2014J|            2013J|         171|
|        CCC|        EEE|            2014J|            2013J|         145|
|        CCC|        EEE|            2014J|            2014B|         113|
|        CCC|        DDD|            2014J|            2014B|         103|
|        CCC|        FFF|            2014J|            2013J|         103|
|        CCC|        FFF|            2014J|            2014B|          91|
|        CCC|        DDD|            2014B|            2013J|          87|
|        CCC|        DDD|            2014B|            2013B|          80|
|        CCC|        FFF|            2014B|            2013J|          56|
|        BBB|        GGG|

In [8]:
# spark.sql('select a.code_module, b.code_module, a.code_presentation, b.code_presentation, count(*) as num_students \
# from studentregistration AS a \
# join studentregistration as b using(id_student) \
# where a.date_unregistration is null and a.code_module < b.code_module and a.code_presentation = b.code_presentation \
# group by a.code_module,  b.code_module, a.code_presentation, b.code_presentation \
# order by num_students DESC').show()

dfs['studentregistration'].select('id_student', 'code_module', 'code_presentation').alias("first").join(dfs['studentregistration'].alias("second"),'id_student', 'inner').where((col('date_unregistration').isNull() & (col('first.code_module') < col('second.code_module')) & (col('first.code_presentation') == col('second.code_presentation')))). \
    groupby(['first.code_module', 'second.code_module', 'first.code_presentation']).count().withColumnRenamed('count', 'num_students').sort(col('num_students').desc()).show()

+-----------+-----------+-----------------+------------+
|code_module|code_module|code_presentation|num_students|
+-----------+-----------+-----------------+------------+
|        CCC|        EEE|            2014B|         217|
|        CCC|        EEE|            2014J|         194|
|        CCC|        DDD|            2014J|          88|
|        CCC|        FFF|            2014J|          71|
|        CCC|        FFF|            2014B|          55|
|        CCC|        DDD|            2014B|          34|
|        DDD|        FFF|            2013J|           4|
|        DDD|        EEE|            2013J|           3|
|        BBB|        GGG|            2014B|           3|
|        DDD|        EEE|            2014B|           2|
|        DDD|        FFF|            2014J|           2|
|        DDD|        FFF|            2014B|           1|
|        BBB|        FFF|            2014J|           1|
|        DDD|        EEE|            2014J|           1|
|        FFF|        GGG|      

In [9]:
# spark.sql('SELECT total.id_student AS id_student, studentinfo.studied_credits AS studied_credits, ROUND(AVG(total.total_score), 2) AS avg_student_score  \
# FROM (total JOIN studentinfo ON ((total.id_student == studentinfo.id_student))) \
# GROUP BY total.id_student \
# ORDER BY total.id_student').show()

In [11]:
# num_males = spark.sql('SELECT COUNT(*) as count FROM studentinfo WHERE gender = "M"').collect()[0]['count']
# num_females = spark.sql('SELECT COUNT(*) as count FROM studentinfo WHERE gender = "F"').collect()[0]['count']
# imd_g_m = spark.sql(f'SELECT imd_band, gender, ROUND(COUNT(*) / {num_males} * 100, 2) AS percentage FROM studentinfo GROUP BY imd_band, gender HAVING gender = "M"')
# imd_g_f = spark.sql(f'SELECT imd_band, gender, ROUND(COUNT(*) / {num_females} * 100, 2) AS percentage FROM studentinfo GROUP BY imd_band, gender HAVING gender = "F"')
# spark.sql('''SELECT *
# FROM imd_g_m
# INNER JOIN imd_g_f USING(imd_band)
# ORDER BY imd_band''').show()

num_males = dfs["studentinfo"].where(col('gender') == 'M').count()
num_females = dfs["studentinfo"].where(col('gender') == 'F').count()

imd_g_m = dfs["studentinfo"].select('imd_band', 'gender').where(col('gender') == 'M').groupby('imd_band', 'gender').agg({'gender': 'count'}).withColumnRenamed('count(gender)', 'count')
imd_g_m = imd_g_m.withColumn('percentage', imd_g_m['count'] / num_males * 100)
imd_g_m = imd_g_m.drop('count')

imd_g_f = dfs["studentinfo"].select('imd_band', 'gender').where(col('gender') == 'F').groupby('imd_band', 'gender').agg({'gender': 'count'}).withColumnRenamed('count(gender)', 'count')
imd_g_f = imd_g_f.withColumn('percentage', imd_g_f['count'] / num_females * 100)
imd_g_f = imd_g_f.drop('count')

imd_g_m.createOrReplaceTempView('imd_g_m')
imd_g_f.createOrReplaceTempView('imd_g_f')
imd_g_m.join(imd_g_f, on='imd_band', how='inner').orderBy('imd_band').show()


+--------+------+------------------+------+------------------+
|imd_band|gender|        percentage|gender|        percentage|
+--------+------+------------------+------+------------------+
|   0-10%|     M|  8.97902097902098|     F|11.591248810979753|
|   10-20|     M| 10.03076923076923|     F|11.706753635004755|
|  20-30%|     M| 10.24895104895105|     F| 12.37939937491507|
|  30-40%|     M|10.858741258741258|     F|10.857453458350319|
|  40-50%|     M| 9.616783216783217|     F|10.442994972142953|
|  50-60%|     M| 9.365034965034965|     F| 9.851882049191467|
|  60-70%|     M| 8.973426573426574|     F| 8.839516238619376|
|  70-80%|     M| 9.454545454545455|     F| 8.078543280337003|
|  80-90%|     M| 9.146853146853147|     F| 7.657290392716401|
| 90-100%|     M| 8.805594405594405|     F| 6.536214159532546|
+--------+------+------------------+------+------------------+



In [137]:
numbers_df = spark.range(1, 1001)
joined = numbers_df.alias('n1').crossJoin(numbers_df.alias('n2')).select(col('n1.id').alias('a'), col('n2.id').alias('b')) \
    .where((col('a') > col('b')))
anticoprimes = joined.crossJoin(numbers_df.alias('n3')).where(
        (col('id') > 1)
        & (col('b') >= col('id'))
        & (col('a') % col('id') == 0)
        & (col('b') % col('id') == 0)) \
    .select('a', 'b')

coprimes = joined.alias('n').join(anticoprimes,on=['a', 'b'], how='leftanti').orderBy(['a', 'b'])
coprimes.show(31)


+---+---+
|  a|  b|
+---+---+
|  2|  1|
|  3|  1|
|  3|  2|
|  4|  1|
|  4|  3|
|  5|  1|
|  5|  2|
|  5|  3|
|  5|  4|
|  6|  1|
|  6|  5|
|  7|  1|
|  7|  2|
|  7|  3|
|  7|  4|
|  7|  5|
|  7|  6|
|  8|  1|
|  8|  3|
|  8|  5|
|  8|  7|
|  9|  1|
|  9|  2|
|  9|  4|
|  9|  5|
|  9|  7|
|  9|  8|
| 10|  1|
| 10|  3|
| 10|  7|
| 10|  9|
+---+---+
only showing top 31 rows

